# Proyecto LOST: experimentos de Q-Learning

Cada ejecución entrena y testea una configuración, guarda su modelo y agrega una fila a `results/q_learning_experiments.csv`. Los resultados quedan disponibles, en orden de registro, para analizarlos manualmente.

In [ ]:
import re
import time
import uuid
from datetime import datetime
from html import escape
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

from experiment_logger import (
    EXPERIMENT_COLUMNS,
    OVERNIGHT_SUMMARY_COLUMNS,
    SEARCH_RUN_COLUMNS,
    append_experiment_result,
    load_experiment_results,
    load_overnight_summary,
    load_search_run_results,
    summarize_test_results,
    summarize_training_history,
)
from q_learning_agent import QLearningAgent

ENV_ID = "MountainCarContinuous-v0"
CSV_PATH = Path("results/q_learning_experiments.csv")
SEARCH_RUNS_CSV_PATH = Path("results/q_learning_search_runs.csv")
OVERNIGHT_SUMMARY_CSV_PATH = Path("results/q_learning_overnight_summary.csv")

## Ejecución y registro

`run_experiment(config, seed=0, notes="")` usa `episodes`, `max_steps` y `test_episodes` como parámetros de la corrida. El resto de la configuración corresponde al agente. Tanto el entrenamiento como el test usan la recompensa original del ambiente.

In [ ]:
def run_experiment(config, seed=0, notes=""):
    config = dict(config)
    if "config_name" not in config:
        raise ValueError("config debe incluir config_name.")

    config_name = str(config.pop("config_name"))
    episodes = int(config.pop("episodes", 50))
    max_steps = int(config.pop("max_steps", 999))
    test_episodes = int(config.pop("test_episodes", 10))
    verbose = bool(config.pop("verbose", False))
    progress_interval = int(config.pop("progress_interval", 500))
    epsilon_start = float(config.pop("epsilon_start", config.pop("epsilon", 1.0)))

    agent_config = {
        "position_bins": 40,
        "velocity_bins": 40,
        "action_bins": 9,
        "alpha": 0.1,
        "gamma": 0.99,
        "epsilon": epsilon_start,
        "epsilon_min": 0.05,
        "epsilon_decay": 0.995,
        "seed": seed,
    }
    configurable_keys = set(agent_config) - {"epsilon", "seed"}
    unknown_keys = set(config) - configurable_keys
    if unknown_keys:
        raise ValueError(f"Parámetros desconocidos: {unknown_keys}")
    agent_config.update(config)

    experiment_id = uuid.uuid4().hex[:12]
    safe_name = re.sub(r"[^a-zA-Z0-9_-]+", "_", config_name).strip("_")
    safe_name = safe_name or "experiment"
    model_path = (
        Path("models")
        / f"q_learning_{safe_name}_seed{seed}_{experiment_id}.pkl"
    )

    env = gym.make(ENV_ID, render_mode="rgb_array")
    agent = QLearningAgent(**agent_config)
    try:
        start_time = time.perf_counter()
        history = agent.train_agent(
            env, episodes=episodes, max_steps=max_steps,
            verbose=verbose, progress_interval=progress_interval,
        )
        training_time = time.perf_counter() - start_time
        test_results = agent.test_agent(
            env, episodes=test_episodes, max_steps=max_steps
        )
    finally:
        env.close()

    agent.save(model_path, metadata={
        "experiment_id": experiment_id, "config_name": config_name,
        "seed": seed, "episodes": episodes, "notes": notes,
    })
    result_row = {
        "experiment_id": experiment_id,
        "timestamp": datetime.now().astimezone().isoformat(timespec="seconds"),
        "config_name": config_name,
        "seed": seed,
        "position_bins": agent_config["position_bins"],
        "velocity_bins": agent_config["velocity_bins"],
        "action_bins": agent.action_bins,
        "alpha": agent_config["alpha"],
        "gamma": agent_config["gamma"],
        "epsilon_start": epsilon_start,
        "epsilon_min": agent_config["epsilon_min"],
        "epsilon_decay": agent_config["epsilon_decay"],
        "episodes": episodes,
        "max_steps": max_steps,
        **summarize_training_history(history),
        "test_episodes": test_episodes,
        **summarize_test_results(test_results),
        "training_time_seconds": round(training_time, 3),
        "model_path": model_path.as_posix(),
        "notes": notes,
        "final_epsilon": agent.epsilon,
    }
    result_row = append_experiment_result(result_row, CSV_PATH)
    print(f"Experimento {experiment_id} registrado en {CSV_PATH}")
    return agent, history, test_results, result_row

## Mi experimento Q-Learning editable

Ejecutá primero las celdas de imports y `run_experiment` que están arriba. Después editá libremente la configuración de la primera celda de esta sección, cambiá `RUN_USER_Q_EXPERIMENT` a `True` en la segunda y ejecutala. El modelo se guarda en `models/` y el resultado se agrega a `results/q_learning_experiments.csv`. El interruptor evita iniciar 10.000 episodios accidentalmente al usar **Run All**.

In [1]:
USER_Q_CONFIG = {
    "config_name": "qlearning_action4_10k",
    "episodes": 10_000,
    "max_steps": 999,
    "test_episodes": 20,
    "position_bins": 40,
    "velocity_bins": 40,
    "action_bins": 4,
    "alpha": 0.1,
    "gamma": 0.99,
    "epsilon": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.999,
    "verbose": True,
    "progress_interval": 500,
}
USER_Q_SEED = 42

USER_Q_CONFIG

{'config_name': 'qlearning_action4_10k',
 'episodes': 10000,
 'max_steps': 999,
 'test_episodes': 20,
 'position_bins': 40,
 'velocity_bins': 40,
 'action_bins': 4,
 'alpha': 0.1,
 'gamma': 0.99,
 'epsilon': 1.0,
 'epsilon_min': 0.05,
 'epsilon_decay': 0.999,
 'verbose': True,
 'progress_interval': 500}

In [2]:
RUN_USER_Q_EXPERIMENT = False

if RUN_USER_Q_EXPERIMENT:
    user_agent, user_history, user_test_results, user_result = run_experiment(
        USER_Q_CONFIG,
        seed=USER_Q_SEED,
        notes="Experimento editable ejecutado desde la notebook",
    )
    result_columns = [
        "config_name", "seed", "episodes", "position_bins",
        "velocity_bins", "action_bins", "alpha", "gamma",
        "epsilon_start", "epsilon_min", "epsilon_decay",
        "test_avg_reward", "test_success_rate",
        "test_avg_max_position", "training_time_seconds", "model_path",
    ]
    display(pd.DataFrame([user_result])[result_columns])
else:
    print("Configuración lista. Cambiá RUN_USER_Q_EXPERIMENT a True para entrenar.")

Configuración lista. Cambiá RUN_USER_Q_EXPERIMENT a True para entrenar.


## Tres configuraciones de ejemplo

Se usan 20 episodios para una validación rápida. Para una corrida real basta cambiar `EXPERIMENT_EPISODES` a 2000, 3000 o 5000 antes de ejecutar esta celda. Cada nueva ejecución agrega filas; nunca reemplaza las anteriores.

In [3]:
EXPERIMENT_EPISODES = 20
TEST_EPISODES = 2

baseline_simple = {
    "config_name": "baseline_simple",
    "position_bins": 30,
    "velocity_bins": 30,
    "action_bins": 7,
    "alpha": 0.10,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.995,
    "episodes": EXPERIMENT_EPISODES,
    "max_steps": 999,
    "test_episodes": TEST_EPISODES,
}

balanced = {
    "config_name": "balanced",
    "position_bins": 40,
    "velocity_bins": 40,
    "action_bins": 9,
    "alpha": 0.15,
    "gamma": 0.99,
    "epsilon_start": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.995,
    "episodes": EXPERIMENT_EPISODES,
    "max_steps": 999,
    "test_episodes": TEST_EPISODES,
}

fine_grained = {
    "config_name": "fine_grained",
    "position_bins": 50,
    "velocity_bins": 50,
    "action_bins": 11,
    "alpha": 0.10,
    "gamma": 0.995,
    "epsilon_start": 1.0,
    "epsilon_min": 0.05,
    "epsilon_decay": 0.997,
    "episodes": EXPERIMENT_EPISODES,
    "max_steps": 999,
    "test_episodes": TEST_EPISODES,
}

In [ ]:
RUN_EXAMPLE_EXPERIMENTS = False

if RUN_EXAMPLE_EXPERIMENTS:
    baseline_run = run_experiment(
        baseline_simple, seed=10, notes="Baseline de grilla compacta"
    )
    balanced_run = run_experiment(
        balanced, seed=20, notes="Mayor resolución y alpha"
    )
    fine_grained_run = run_experiment(
        fine_grained, seed=30, notes="Grilla fina y decay más lento"
    )
else:
    print("Ejemplos omitidos. Cambiá RUN_EXAMPLE_EXPERIMENTS a True para entrenarlos.")

## Análisis de búsqueda overnight

Esta sección detecta exclusivamente la última búsqueda overnight completa: perfil `overnight`, 20.000 episodios, 100 tests y 60 entrenamientos completados. Las validaciones cortas y búsquedas anteriores quedan excluidas.

Estas gráficas no seleccionan automáticamente un modelo. El objetivo es comparar manualmente configuraciones. Como casi todas alcanzan 100% de éxito, la comparación útil se hace por recompensa, pasos, estabilidad y tiempo. Una configuración robusta debería tener alta recompensa, pocos pasos, baja desviación entre seeds y tiempo razonable.

In [ ]:
EXPERIMENTS_CSV = Path("results/q_learning_experiments.csv")
SEARCH_RUNS_CSV = Path("results/q_learning_search_runs.csv")
OVERNIGHT_SUMMARY_CSV = Path("results/q_learning_overnight_summary.csv")

experiments_df = pd.read_csv(EXPERIMENTS_CSV)
search_runs_df = pd.read_csv(SEARCH_RUNS_CSV)
overnight_summary_df = pd.read_csv(OVERNIGHT_SUMMARY_CSV)

overnight_candidates = search_runs_df.loc[
    search_runs_df["profile"].eq("overnight")
    & search_runs_df["episodes"].eq(20000)
    & search_runs_df["test_episodes"].eq(100)
    & search_runs_df["status"].eq("completed")
    & search_runs_df["number_of_configs_completed"].eq(60)
].copy()
if overnight_candidates.empty:
    raise ValueError("No se encontró una búsqueda overnight completa.")

date_column = "completed_at" if "completed_at" in overnight_candidates else "timestamp_end"
overnight_candidates["_completion_time"] = pd.to_datetime(
    overnight_candidates[date_column], errors="coerce"
)
overnight_candidates = overnight_candidates.sort_values(
    "_completion_time", kind="stable"
)
overnight_run = overnight_candidates.iloc[-1]
OVERNIGHT_SEARCH_ID = str(overnight_run["search_id"])

selected_experiments_df = experiments_df.loc[
    experiments_df["search_id"].astype(str).eq(OVERNIGHT_SEARCH_ID)
    & experiments_df["profile"].eq("overnight")
    & experiments_df["episodes"].eq(20000)
    & experiments_df["test_episodes"].eq(100)
].copy()
selected_summary_df = overnight_summary_df.loc[
    overnight_summary_df["search_id"].astype(str).eq(OVERNIGHT_SEARCH_ID)
    & overnight_summary_df["profile"].eq("overnight")
].copy()

if len(selected_experiments_df) != 60 or selected_summary_df.empty:
    raise ValueError("Los CSV no contienen los 60 experimentos y su resumen.")

search_info_columns = [
    "search_id", "search_name", "status",
    "number_of_configs_completed", "total_search_time_seconds",
]
display(overnight_run[search_info_columns].to_frame("valor"))
print(f"Configuraciones resumidas: {len(selected_summary_df)}")
print(f"Experimentos individuales: {len(selected_experiments_df)}")

In [ ]:
detail_columns = [
    "config_name", "action_bins", "alpha", "gamma",
    "epsilon_start", "epsilon_min", "epsilon_decay",
    "position_bins", "velocity_bins",
]
config_details_df = selected_experiments_df[detail_columns].drop_duplicates(
    subset="config_name", keep="first"
)
comparison_df = selected_summary_df.merge(
    config_details_df, on="config_name", how="left", validate="one_to_one"
)

principal_columns = [
    "config_name", "seeds_completed",
    "mean_test_success_rate", "std_test_success_rate",
    "mean_test_avg_reward", "std_test_avg_reward",
    "mean_test_avg_steps", "std_test_avg_steps",
    "mean_test_avg_max_position", "mean_training_time_seconds",
    "mean_train_success_rate_last_100",
    "mean_train_last_100_avg_env_reward",
    "action_bins", "alpha", "gamma",
    "epsilon_start", "epsilon_min", "epsilon_decay",
    "position_bins", "velocity_bins",
]
principal_columns = [column for column in principal_columns if column in comparison_df]
principal_table = comparison_df[principal_columns].copy()
numeric_columns = principal_table.select_dtypes(include="number").columns
principal_table[numeric_columns] = principal_table[numeric_columns].round(3)
display(principal_table)

In [ ]:
def plot_horizontal_bar(data, value_column, title, xlabel, ascending, error_column=None, color="steelblue"):
    ordered = data.sort_values(value_column, ascending=ascending, kind="stable")
    errors = ordered[error_column] if error_column else None
    fig, ax = plt.subplots(figsize=(11, 7))
    bars = ax.barh(
        ordered["config_name"], ordered[value_column],
        xerr=errors, capsize=3, color=color, alpha=0.85,
    )
    ax.invert_yaxis()
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", alpha=0.25)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    fig.tight_layout()
    plt.show()


plot_horizontal_bar(
    comparison_df, "mean_test_avg_reward",
    "Recompensa promedio en test por configuración",
    "Recompensa promedio (reward original)", ascending=False,
    error_column="std_test_avg_reward", color="seagreen",
)
plot_horizontal_bar(
    comparison_df, "mean_test_avg_steps",
    "Pasos promedio hasta la meta en test",
    "Pasos promedio", ascending=True,
    error_column="std_test_avg_steps", color="cornflowerblue",
)
plot_horizontal_bar(
    comparison_df, "mean_training_time_seconds",
    "Tiempo promedio de entrenamiento por configuración",
    "Segundos", ascending=True, color="darkorange",
)

## Estabilidad y trade-offs

Las desviaciones permiten evaluar estabilidad entre seeds. En el gráfico recompensa vs pasos, arriba significa mayor recompensa e izquierda significa menos pasos: las configuraciones más atractivas están arriba a la izquierda. El segundo scatter muestra el compromiso entre recompensa y costo de entrenamiento. El orden visual de las barras sirve solo para leer mejor cada métrica; no constituye una elección automática.

In [ ]:
plot_horizontal_bar(
    comparison_df, "std_test_avg_reward",
    "Variabilidad de recompensa entre seeds",
    "Desviación estándar de recompensa", ascending=True, color="mediumpurple",
)
plot_horizontal_bar(
    comparison_df, "std_test_avg_steps",
    "Variabilidad de pasos entre seeds",
    "Desviación estándar de pasos", ascending=True, color="slateblue",
)

def plot_tradeoff(data, x_column, y_column, title, xlabel, ylabel):
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.scatter(data[x_column], data[y_column], s=75, alpha=0.8)
    for row in data.itertuples(index=False):
        ax.annotate(
            row.config_name,
            (getattr(row, x_column), getattr(row, y_column)),
            xytext=(5, 5), textcoords="offset points", fontsize=8,
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    fig.tight_layout()
    plt.show()


plot_tradeoff(
    comparison_df, "mean_test_avg_steps", "mean_test_avg_reward",
    "Trade-off: recompensa vs pasos", "Pasos promedio", "Recompensa promedio",
)
plot_tradeoff(
    comparison_df, "mean_training_time_seconds", "mean_test_avg_reward",
    "Trade-off: recompensa vs tiempo de entrenamiento",
    "Tiempo promedio de entrenamiento (s)", "Recompensa promedio",
)

## Tasa de éxito como control

Cuando varias configuraciones tienen 100% de éxito, esta métrica deja de diferenciar. Para decidir conviene mirar recompensa, pasos, estabilidad y tiempo.

In [ ]:
success_control_df = comparison_df[[
    "config_name", "mean_test_success_rate", "std_test_success_rate"
]].copy()
success_control_df[["mean_test_success_rate", "std_test_success_rate"]] = (
    success_control_df[["mean_test_success_rate", "std_test_success_rate"]].round(3)
)
display(success_control_df)

## Análisis por seed de una configuración elegida manualmente

Editá `CHOSEN_CONFIG_NAME` para inspeccionar otra configuración. La tabla y los gráficos usan únicamente sus cinco seeds dentro de la búsqueda overnight completa detectada arriba. Esta celda no elige un modelo.

In [ ]:
PREFERRED_CONFIG_NAME = "baseline_40x40_a11"
available_configs = sorted(selected_experiments_df["config_name"].unique())
CHOSEN_CONFIG_NAME = (
    PREFERRED_CONFIG_NAME if PREFERRED_CONFIG_NAME in available_configs
    else available_configs[0]
)

chosen_seed_df = selected_experiments_df.loc[
    selected_experiments_df["config_name"].eq(CHOSEN_CONFIG_NAME),
    [
        "seed", "test_success_rate", "test_avg_reward",
        "test_avg_steps", "test_avg_max_position",
        "training_time_seconds", "model_path",
    ],
].sort_values("seed", kind="stable")
if chosen_seed_df.empty:
    available = sorted(selected_experiments_df["config_name"].unique())
    raise ValueError(f"Configuración desconocida. Opciones: {available}")

display(chosen_seed_df.reset_index(drop=True))

seed_labels = chosen_seed_df["seed"].astype(str)
seed_metrics = [
    ("test_avg_reward", "Recompensa de test por seed", "Recompensa", "seagreen"),
    ("test_avg_steps", "Pasos de test por seed", "Pasos", "cornflowerblue"),
    ("training_time_seconds", "Tiempo de entrenamiento por seed", "Segundos", "darkorange"),
]
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for axis, (column, title, ylabel, color) in zip(axes, seed_metrics):
    bars = axis.bar(seed_labels, chosen_seed_df[column], color=color, alpha=0.85)
    axis.set_title(title)
    axis.set_xlabel("Seed")
    axis.set_ylabel(ylabel)
    axis.grid(axis="y", alpha=0.25)
    axis.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
fig.suptitle(f"Detalle por seed: {CHOSEN_CONFIG_NAME}")
fig.tight_layout()
plt.show()